# Clasificación de imágenes satelitales mediante redes neuronales convolucionales

**TensorFlow y Keras | EuroSAT RGB**

El trabajo compara dos arquitecturas convolucionales para identificar categorías de cobertura terrestre a partir de imágenes satelitales. Se incluye exploración de los datos, entrenamiento, ajuste de la arquitectura, evaluación y una función de predicción.


## Entorno de trabajo

Se utilizaron Python, TensorFlow, Keras y las bibliotecas necesarias para el manejo de datos y la evaluación de modelos.

In [ ]:
import sys
import subprocess
import importlib.util

for modulo, paquete in [
    ("importlib_resources", "importlib-resources"),
    ("datasets", "datasets"),
]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
BATCH_SIZE = 64
EPOCAS_BASE = 10
EPOCAS_MEJORADO = 15

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 1. Introducción

### Planteamiento del problema

La clasificación de cobertura terrestre a partir de imágenes satelitales es útil para el estudio del territorio, el monitoreo ambiental y el análisis del crecimiento urbano. Automatizar esta tarea permite procesar colecciones de imágenes de manera consistente.

Se plantea un problema de clasificación multiclase: dada una imagen RGB de 64 × 64 píxeles, identificar una de diez categorías de cobertura terrestre.

### Objetivo

Desarrollar un clasificador basado en redes neuronales convolucionales (CNN) y analizar su desempeño mediante un conjunto independiente de prueba. También se compara una arquitectura inicial con una segunda versión que incorpora cambios de regularización y de estructura.

### Conjunto de datos

Se utilizó **EuroSAT RGB**, compuesto por 27,000 imágenes de Sentinel-2, con diez clases etiquetadas. Las imágenes tienen tres canales de color y una resolución de 64 × 64 píxeles.

Fuente de los datos: https://huggingface.co/datasets/giswqs/EuroSAT_RGB  
Proyecto original: https://github.com/phelber/eurosat

Referencia: Helber, P., Bischke, B., Dengel, A. y Borth, D. (2019). *EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification*. IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing, 12(7), 2217–2226. https://doi.org/10.1109/JSTARS.2019.2918242

## 2. Datos y preprocesamiento

### 2.1 Carga de datos

El conjunto de datos se obtuvo desde Hugging Face. Se reunieron las particiones disponibles para crear una división estratificada común a los dos experimentos.

In [ ]:
FUENTE_DATOS = "giswqs/EuroSAT_RGB"

particiones = load_dataset(FUENTE_DATOS)
datos = concatenate_datasets([particiones[n] for n in particiones.keys()])

nombres_clases = datos.features["label"].names
num_clases = len(nombres_clases)

print("Imágenes:", len(datos))
print("Clases:", num_clases)
print("Categorías:", nombres_clases)

assert len(datos) == 27000
assert num_clases == 10


### 2.2 Características de las imágenes

Las imágenes se transformaron en arreglos NumPy de tipo `uint8`. Cada observación contiene 64 × 64 píxeles y tres canales RGB.

In [ ]:
X = np.empty((len(datos), 64, 64, 3), dtype=np.uint8)
y = np.empty(len(datos), dtype=np.int64)

for i, ejemplo in enumerate(datos):
    X[i] = np.asarray(ejemplo["image"].convert("RGB"), dtype=np.uint8)
    y[i] = int(ejemplo["label"])

del datos, particiones

print("Dimensiones:", X.shape)
print("Valores de píxel:", X.min(), "a", X.max())
print("Etiquetas:", np.unique(y))
assert X.shape == (27000, 64, 64, 3)


### 2.3 Distribución de clases

Se revisó la cantidad de observaciones por categoría para identificar posibles diferencias en su representación.

In [ ]:
conteos = np.bincount(y, minlength=num_clases)
distribucion = pd.DataFrame({
    "Categoría": nombres_clases,
    "Cantidad": conteos,
    "Porcentaje": np.round(100 * conteos / len(y), 2)
}).sort_values("Cantidad", ascending=False)
print(distribucion.to_string(index=False))

plt.figure(figsize=(11, 4))
plt.bar(distribucion["Categoría"], distribucion["Cantidad"])
plt.title("Cantidad de imágenes por categoría")
plt.ylabel("Número de imágenes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 2.4 Ejemplos del conjunto de datos

Se muestra una imagen representativa de cada clase.

In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(13, 6))
for clase_id, ax in enumerate(ejes.ravel()):
    idx = int(np.flatnonzero(y == clase_id)[0])
    ax.imshow(X[idx])
    ax.set_title(nombres_clases[clase_id])
    ax.axis("off")
plt.suptitle("Una imagen por categoría", y=1.02)
plt.tight_layout()
plt.show()


### 2.5 Conjuntos de entrenamiento, validación y prueba

La división se realizó de forma estratificada: 70% de las imágenes para entrenamiento, 15% para validación y 15% para prueba. La validación se utilizó para comparar las arquitecturas; el conjunto de prueba se reservó para evaluar el modelo seleccionado.

In [ ]:
X_train, X_temporal, y_train, y_temporal = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temporal, y_temporal, test_size=0.50, random_state=SEED, stratify=y_temporal
)
del X, y, X_temporal, y_temporal

for nombre, imagenes, etiquetas in [
    ("Entrenamiento", X_train, y_train),
    ("Validación", X_val, y_val),
    ("Prueba", X_test, y_test)
]:
    print(f"{nombre}: {len(etiquetas):,} imágenes ({len(etiquetas)/27000:.1%})")
    assert len(np.unique(etiquetas)) == num_clases

assert len(X_train) == 18900
assert len(X_val) == 4050
assert len(X_test) == 4050


In [ ]:
filas = []
for i, nombre in enumerate(nombres_clases):
    filas.append({
        "Categoría": nombre,
        "Entrenamiento": int(np.sum(y_train == i)),
        "Validación": int(np.sum(y_val == i)),
        "Prueba": int(np.sum(y_test == i))
    })
print(pd.DataFrame(filas).to_string(index=False))


### 2.6 Preparación de los datos

Las imágenes se agruparon en lotes de 64 observaciones. El orden se aleatorizó únicamente en entrenamiento. La normalización de los valores de píxeles al intervalo [0, 1] se realizó con una capa `Rescaling` dentro de cada modelo.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=5000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

lote_imagenes, lote_etiquetas = next(iter(train_ds))
print("Forma de un lote de imágenes:", lote_imagenes.shape)
print("Forma de un lote de etiquetas:", lote_etiquetas.shape)
print("Tipo de imágenes:", lote_imagenes.dtype)


## 3. Desarrollo del modelo inicial

### 3.1 Arquitectura

Se construyó una CNN con tres capas convolucionales `Conv2D`, cada una seguida de `MaxPooling2D`. Después se utilizaron `Flatten` y dos capas densas para obtener una distribución de probabilidad sobre las diez clases.

La función de pérdida fue `sparse_categorical_crossentropy`, dado que las etiquetas se representan mediante enteros. Se utilizó Adam como optimizador y `accuracy` como métrica principal.

In [ ]:
def crear_cnn_base():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(num_clases, activation="softmax")
    ], name="cnn_base")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

modelo_base = crear_cnn_base()
modelo_base.summary()


### 3.2 Entrenamiento

Se entrenó el modelo con parada temprana basada en la pérdida de validación. El mejor estado de los pesos se conservó en un archivo de Keras.

In [ ]:
callbacks_base = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "modelo_base.keras", monitor="val_loss", save_best_only=True
    )
]

historia_base = modelo_base.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCAS_BASE,
    callbacks=callbacks_base,
    verbose=1
)
print("Épocas realizadas (base):", len(historia_base.history["loss"]))


### 3.3 Curvas de aprendizaje

Se compararon la precisión y la pérdida registradas durante el entrenamiento y la validación.

In [ ]:
def graficar_historial(historia, titulo):
    h = historia.history
    epocas = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epocas, h["accuracy"], label="Entrenamiento")
    ax[0].plot(epocas, h["val_accuracy"], label="Validación")
    ax[0].set(title=f"{titulo}: accuracy", xlabel="Época", ylabel="Accuracy")
    ax[0].legend()
    ax[1].plot(epocas, h["loss"], label="Entrenamiento")
    ax[1].plot(epocas, h["val_loss"], label="Validación")
    ax[1].set(title=f"{titulo}: loss", xlabel="Época", ylabel="Loss")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

graficar_historial(historia_base, "Modelo inicial")


In [ ]:
modelo_base = tf.keras.models.load_model("modelo_base.keras")
metricas_base_val = modelo_base.evaluate(val_ds, return_dict=True, verbose=0)
print("Validación: CNN inicial")
for metrica, valor in metricas_base_val.items():
    print(f"{metrica}: {valor:.4f}")


## 4. Ajuste del modelo

### 4.1 Cambios en la arquitectura

La segunda versión incorporó aumento de datos mediante transformaciones geométricas, `BatchNormalization` y `Dropout`. También se sustituyó `Flatten` por `GlobalAveragePooling2D` para reducir el número de parámetros de la etapa de clasificación.

Se utilizó `ReduceLROnPlateau` para disminuir la tasa de aprendizaje cuando la pérdida de validación dejara de mejorar. Estos cambios se evaluaron de manera conjunta; el experimento permite comparar las dos configuraciones, pero no aislar el efecto de cada técnica.

In [ ]:
def crear_cnn_mejorada():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3)),
        tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        tf.keras.layers.RandomRotation(0.10, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.Rescaling(1.0 / 255),

        tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.15),

        tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.20),

        tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(num_clases, activation="softmax")
    ], name="cnn_mejorada")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

modelo_mejorado = crear_cnn_mejorada()
modelo_mejorado.summary()
print("Parámetros del modelo base:", modelo_base.count_params())
print("Parámetros del modelo mejorado:", modelo_mejorado.count_params())


### 4.2 Entrenamiento de la segunda versión

Se mantuvieron los mismos conjuntos de datos, la función de pérdida y la métrica de evaluación del experimento inicial.

In [ ]:
callbacks_mejorado = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2,
        min_lr=1e-5, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "modelo_mejorado.keras", monitor="val_loss", save_best_only=True
    )
]

historia_mejorado = modelo_mejorado.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCAS_MEJORADO,
    callbacks=callbacks_mejorado,
    verbose=1
)
print("Épocas realizadas (ajustado):", len(historia_mejorado.history["loss"]))


In [ ]:
graficar_historial(historia_mejorado, "Modelo mejorado")


### 4.3 Comparación de modelos

La selección se realizó según la precisión de validación. En caso de empate se consideró la menor pérdida de validación. El conjunto de prueba no se utilizó para seleccionar el modelo.

In [ ]:
modelo_mejorado = tf.keras.models.load_model("modelo_mejorado.keras")
metricas_mejorado_val = modelo_mejorado.evaluate(val_ds, return_dict=True, verbose=0)

comparacion = pd.DataFrame([
    {
        "Modelo": "CNN inicial",
        "Accuracy validación": metricas_base_val["accuracy"],
        "Loss validación": metricas_base_val["loss"],
        "Parámetros": modelo_base.count_params(),
        "Épocas": len(historia_base.history["loss"])
    },
    {
        "Modelo": "CNN ajustada",
        "Accuracy validación": metricas_mejorado_val["accuracy"],
        "Loss validación": metricas_mejorado_val["loss"],
        "Parámetros": modelo_mejorado.count_params(),
        "Épocas": len(historia_mejorado.history["loss"])
    }
])
display(comparacion.round(4))

ranking = comparacion.sort_values(
    ["Accuracy validación", "Loss validación"], ascending=[False, True]
)
nombre_ganador = ranking.iloc[0]["Modelo"]
modelo_final = modelo_mejorado if nombre_ganador == "CNN ajustada" else modelo_base

diferencia = metricas_mejorado_val["accuracy"] - metricas_base_val["accuracy"]
print("Modelo seleccionado:", nombre_ganador)
print(f"Diferencia de accuracy (ajustada - inicial): {100*diferencia:+.2f} puntos porcentuales")


## 5. Resultados

El modelo seleccionado se evaluó con las imágenes de prueba. Se reportaron la precisión global (*accuracy*), la pérdida, el F1 macro y las métricas por categoría. También se construyó una matriz de confusión para analizar los errores de clasificación.

In [ ]:
metricas_test = modelo_final.evaluate(test_ds, return_dict=True, verbose=0)
probabilidades = modelo_final.predict(test_ds, verbose=0)
predicciones = np.argmax(probabilidades, axis=1)
verdaderas = y_test.copy()

f1_macro = f1_score(verdaderas, predicciones, average="macro", zero_division=0)
print("MODELO FINAL:", nombre_ganador)
print(f"Accuracy en prueba: {metricas_test['accuracy']:.4f}")
print(f"Loss en prueba: {metricas_test['loss']:.4f}")
print(f"F1 macro en prueba: {f1_macro:.4f}")
print("\nREPORTE POR CATEGORÍA")
print(classification_report(
    verdaderas, predicciones,
    labels=np.arange(num_clases),
    target_names=nombres_clases,
    digits=3, zero_division=0
))


In [ ]:
matriz = confusion_matrix(
    verdaderas, predicciones, labels=np.arange(num_clases)
)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matriz, cmap="Blues")
ax.set_xticks(range(num_clases), nombres_clases, rotation=55, ha="right")
ax.set_yticks(range(num_clases), nombres_clases)
ax.set_xlabel("Predicción")
ax.set_ylabel("Etiqueta real")
ax.set_title(f"Matriz de confusión — {nombre_ganador}")
for i in range(num_clases):
    for j in range(num_clases):
        valor = matriz[i, j]
        ax.text(j, i, str(valor), ha="center", va="center",
                color="white" if valor > matriz.max() / 2 else "black", fontsize=8)
plt.colorbar(im, ax=ax, label="Número de imágenes")
plt.tight_layout()
plt.show()


In [ ]:
reporte = classification_report(
    verdaderas, predicciones, labels=np.arange(num_clases),
    target_names=nombres_clases, output_dict=True, zero_division=0
)
f1_clases = pd.DataFrame([
    {"Categoría": clase, "F1": reporte[clase]["f1-score"],
     "Recall": reporte[clase]["recall"], "Precision": reporte[clase]["precision"]}
    for clase in nombres_clases
]).sort_values("F1")
print("Categorías ordenadas por F1 (más difíciles primero):")
print(f1_clases.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

plt.figure(figsize=(10, 4))
plt.bar(f1_clases["Categoría"], f1_clases["F1"])
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.title("F1-score por categoría (prueba)")
plt.ylabel("F1-score")
plt.tight_layout()
plt.show()


### 5.1 Ejemplos de predicciones

Se muestran ejemplos de aciertos y errores de clasificación junto con la probabilidad asignada a la clase predicha.

In [ ]:
indices_fallos = np.flatnonzero(predicciones != verdaderas)[:6]
indices_aciertos = np.flatnonzero(predicciones == verdaderas)[:6]

fig, ejes = plt.subplots(2, 6, figsize=(16, 6))
for fila, indices in enumerate([indices_fallos, indices_aciertos]):
    for columna, idx in enumerate(indices):
        ax = ejes[fila, columna]
        ax.imshow(X_test[idx])
        ax.set_title(
            f"Real: {nombres_clases[verdaderas[idx]]}\n"
            f"Pred.: {nombres_clases[predicciones[idx]]}\n"
            f"Prob.: {probabilidades[idx].max():.1%}",
            fontsize=8
        )
        ax.axis("off")
    for columna in range(len(indices), 6):
        ejes[fila, columna].axis("off")

ejes[0, 0].set_ylabel("Errores")
ejes[1, 0].set_ylabel("Aciertos")
plt.tight_layout()
plt.show()


## 6. Conclusiones

El análisis considera el desempeño de las dos versiones en validación y los resultados del modelo seleccionado en prueba. Las diferencias observadas deben interpretarse como un efecto conjunto de las modificaciones realizadas.

El uso de imágenes RGB de 64 × 64 píxeles limita el detalle disponible para distinguir algunas categorías. Además, la división aleatoria permite evaluar el rendimiento sobre muestras del mismo conjunto, pero no garantiza el mismo desempeño en otras regiones geográficas.

Como trabajo futuro, se propone evaluar las técnicas de regularización por separado, considerar aprendizaje por transferencia y utilizar una partición geográficamente independiente.

In [ ]:
from IPython.display import Markdown, display

peor_clase = f1_clases.iloc[0]
mejor_clase = f1_clases.iloc[-1]
matriz_sin_diagonal = matriz.copy()
np.fill_diagonal(matriz_sin_diagonal, 0)
fila_error, col_error = np.unravel_index(
    np.argmax(matriz_sin_diagonal), matriz_sin_diagonal.shape
)

cambio = "superó" if diferencia > 0 else "quedó por debajo de" if diferencia < 0 else "igualó"
comparacion_texto = (
    f"La CNN ajustada {cambio} la precisión de validación de la CNN inicial "
    f"(diferencia de {abs(diferencia)*100:.2f} puntos porcentuales)."
)

conclusion = f"""
**Síntesis de resultados.** {comparacion_texto} El modelo seleccionado fue **{nombre_ganador}**. 
En prueba obtuvo una precisión de **{metricas_test['accuracy']:.2%}** y un F1 macro de **{f1_macro:.3f}**.

La categoría con menor F1 fue **{peor_clase['Categoría']}** ({peor_clase['F1']:.3f}), 
mientras que **{mejor_clase['Categoría']}** presentó el mayor F1 ({mejor_clase['F1']:.3f}).
"""
if matriz_sin_diagonal[fila_error, col_error] > 0:
    conclusion += (
        f" La confusión más frecuente ocurrió entre **{nombres_clases[fila_error]}** "
        f"(real) y **{nombres_clases[col_error]}** (predicha), "
        f"con **{matriz_sin_diagonal[fila_error, col_error]}** casos."
    )

display(Markdown(conclusion))


## 7. Aplicación del modelo

### 7.1 Almacenamiento

El modelo seleccionado se exportó en formato `.keras`, que conserva la arquitectura y los pesos entrenados. Las métricas y los nombres de las clases se guardaron en un archivo JSON.

In [ ]:
ruta_modelo = "modelo_final.keras"
modelo_final.save(ruta_modelo)

reporte_final = {
    "dataset": FUENTE_DATOS,
    "seed": SEED,
    "train_samples": len(y_train),
    "val_samples": len(y_val),
    "test_samples": len(y_test),
    "selected_model": nombre_ganador,
    "validation_baseline": {k: float(v) for k, v in metricas_base_val.items()},
    "validation_improved": {k: float(v) for k, v in metricas_mejorado_val.items()},
    "test": {**{k: float(v) for k, v in metricas_test.items()}, "f1_macro": float(f1_macro)},
    "classes": nombres_clases
}
with open("metricas_finales.json", "w", encoding="utf-8") as archivo:
    json.dump(reporte_final, archivo, indent=2, ensure_ascii=False)

print("Modelo guardado:", os.path.abspath(ruta_modelo))
print("Tamaño (MB):", round(os.path.getsize(ruta_modelo) / (1024**2), 2))
print("Métricas guardadas: metricas_finales.json")


### 7.2 Verificación del modelo guardado

Se compararon las predicciones del modelo original y del modelo recuperado desde el archivo `.keras`.

In [ ]:
modelo_recargado = tf.keras.models.load_model("modelo_final.keras")
original = modelo_final.predict(X_test[:5], verbose=0)
recargado = modelo_recargado.predict(X_test[:5], verbose=0)
np.testing.assert_allclose(original, recargado, rtol=1e-4, atol=1e-5)
print("OK: el modelo .keras se recargó y reproduce las predicciones.")
print("Categorías predichas:", [nombres_clases[i] for i in np.argmax(recargado, axis=1)])


### 7.3 Clasificación de imágenes

La función `clasificar_imagen(ruta)` recibe la ruta de una imagen, la ajusta a 64 × 64 píxeles y devuelve las tres categorías con mayor probabilidad. Las probabilidades corresponden a la salida `softmax` y no constituyen, por sí mismas, una medida calibrada de confianza.

In [ ]:
def clasificar_imagen(ruta, modelo=modelo_recargado):
    imagen = tf.keras.utils.load_img(
        ruta, target_size=(64, 64), color_mode="rgb"
    )
    arr = tf.keras.utils.img_to_array(imagen)
    probabilidades_imagen = modelo.predict(
        np.expand_dims(arr, axis=0), verbose=0
    )[0]
    indices = np.argsort(probabilidades_imagen)[::-1][:3]
    resultado = [(nombres_clases[i], float(probabilidades_imagen[i])) for i in indices]

    plt.figure(figsize=(4, 4))
    plt.imshow(imagen)
    plt.title(resultado[0][0])
    plt.axis("off")
    plt.show()

    print("Tres clases con mayor probabilidad:")
    for clase, probabilidad in resultado:
        print(f"{clase}: {probabilidad:.2%}")
    return resultado

ruta_ejemplo = "ejemplo_eurosat.png"
tf.keras.utils.save_img(ruta_ejemplo, X_test[0])
resultado_ejemplo = clasificar_imagen(ruta_ejemplo)
print("Categoría real:", nombres_clases[y_test[0]])
